# Download and process Sentinel-2 L2A products to CPM and upload to S3

One-off script, using command-line interfaces of third-party software

## Setup

1. Install `s5cmd`
2. Set up your `.aws/credentials` and `.aws/config` with profiles for CDSE and your target S3 configuration
3. Install [eopf-cpm](https://gitlab.eopf.copernicus.eu/cpm/eopf-cpm) and [eopf-stac](https://github.com/EOPF-Sample-Service/eopf-stac) (nb: requires [this revision](https://github.com/EOPF-Sample-Service/eopf-stac/pull/65))

## Usage

1. Adapt the CDSE STAC query parameters to your needs
2. Change the target S3 configuration, if necessary
3. Run script top-to-bottom - NB: no retry logic
4. Note that the STAC JSON files are not deleted locally after upload and used to identify products that are completed

In [8]:
import shutil
import subprocess
from pathlib import Path

from pystac_client import Client

In [9]:
catalog = Client.open("https://stac.dataspace.copernicus.eu/v1/")

In [10]:
# Define your temporal range
daterange = {"interval": ["2025-08-10T00:00:00Z", "2025-08-31T23:59:59Z"]}

# Define your search with CQL2 syntax
search = catalog.search(
    filter_lang="cql2-json",
    filter={
        "op": "and",
        "args": [
            {"op": "anyinteracts", "args": [{"property": "datetime"}, daterange]},
            {"op": "=", "args": [{"property": "collection"}, "sentinel-2-l2a"]},
            {"op": "=", "args": [{"property": "grid:code"}, "MGRS-29TPF"]},
        ],
    },
)

In [11]:
items = list(search.items())
len(items)

15

In [12]:
CDSE_S3_ENDPOINT_URL = "https://eodata.dataspace.copernicus.eu/"
CDSE_S3_PROFILE = "cdse"
TARGET_S3_ENDPOINT_URL = "https://s3.de.io.cloud.ovh.net/"
TARGET_S3_PROFILE = "eopf-explorer"
TARGET_PREFIX = "s3://esa-zarr-sentinel-explorer-fra/cpm-manual/"  # keep trailing slash
TARGET_ASSET_HREF_BASE = "https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-fra/cpm-manual/"  # keep trailing slash

In [13]:
def get_safe_root(item):
    asset_href = next(iter(item.assets.values())).href
    return asset_href.split(".SAFE")[0] + ".SAFE"

In [ ]:
ROOT_DIR = Path("./wildfire-pt-2025")

for item in items:
    prefix = get_safe_root(item)
    print(prefix)

    safe_dir = ROOT_DIR / "safe" / Path(prefix).name
    cpm_dir = ROOT_DIR / "cpm" / (Path(prefix).stem + ".zarr")
    stac_file = cpm_dir.parent / cpm_dir.with_suffix(".json").name
    cmp_uri = f"{TARGET_PREFIX}{cpm_dir.name}/"
    stac_uri = f"{TARGET_PREFIX}{stac_file.name}"
    cmp_http_uri = f"{TARGET_ASSET_HREF_BASE}{cpm_dir.name}/"

    print(cpm_dir)
    print(cmp_http_uri)

    # # Check whether local product exists
    if stac_file.exists():
        print(f"STAC file {stac_file} exists. Skipping.")
        continue

    # Check whether remote product exists
    if False:
        try:
            subprocess.run(
                [
                    "s5cmd",
                    "--profile",
                    TARGET_S3_PROFILE,
                    "--endpoint-url",
                    TARGET_S3_ENDPOINT_URL,
                    "head",
                    stac_uri,
                ],
                check=True,
            )
            print(f"STAC file {stac_uri} exists. Skipping.")
            continue
        except Exception:
            pass

    # Download SAFE from CDSE
    if not safe_dir.exists():
        print(f"Downloading {safe_dir.name}")
        safe_dir.mkdir(parents=True)
        try:
            subprocess.run(
                [
                    "s5cmd",
                    "--profile",
                    CDSE_S3_PROFILE,
                    "--endpoint-url",
                    CDSE_S3_ENDPOINT_URL,
                    "sync",
                    f"{prefix}/*",
                    str(safe_dir.absolute()),
                ],
                check=True,
            )
        except Exception:
            shutil.rmtree(safe_dir)
            raise

    # Produce CPM Zarr
    if not cpm_dir.exists():
        print(f"Producing {cpm_dir.name}")
        cpm_dir.mkdir(parents=True)
        try:
            subprocess.run(["eopf", "convert", str(safe_dir), str(cpm_dir)], check=True)
            shutil.rmtree(safe_dir)
        except Exception:
            shutil.rmtree(cpm_dir)
            raise

    # Produce STAC item metadata
    if not stac_file.exists():
        print(f"Producing {stac_file.name}")
        subprocess.run(
            [
                "eopf-stac",
                str(cpm_dir),
                "--output-file",
                str(stac_file),
                # "--asset-href-base", TARGET_ASSET_HREF_BASE,
            ],
            check=True,
        )

    # Substitute asset hrefs in the STAC file to point to the target S3 location (replace the local path with the target S3 URI)
    with open(stac_file) as f:
        stac_content = f.read()
    stac_content = stac_content.replace(str(cpm_dir), cmp_http_uri)
    with open(stac_file, "w") as f:
        f.write(stac_content)

    # Upload CPM Zarr product
    print(f"Uploading {cpm_dir.name}")
    subprocess.run(
        [
            "s5cmd",
            "--profile",
            TARGET_S3_PROFILE,
            "--endpoint-url",
            TARGET_S3_ENDPOINT_URL,
            "sync",
            f"{cpm_dir.absolute()}/",
            cmp_uri,
        ],
        check=True,
    )
    subprocess.run(
        [
            "s5cmd",
            "--profile",
            TARGET_S3_PROFILE,
            "--endpoint-url",
            TARGET_S3_ENDPOINT_URL,
            "cp",
            str(stac_file.absolute()),
            stac_uri,
        ],
        check=True,
    )
    # Delete the CMP dir, keep the local STAC file as a token of completion
    shutil.rmtree(cpm_dir)

s3://eodata/Sentinel-2/MSI/L2A/2025/08/31/S2C_MSIL2A_20250831T112131_N0511_R037_T29TPF_20250831T154413.SAFE
wildfire-pt-2025/cpm/S2C_MSIL2A_20250831T112131_N0511_R037_T29TPF_20250831T154413.zarr
https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-fra/cpm-manual/S2C_MSIL2A_20250831T112131_N0511_R037_T29TPF_20250831T154413.zarr/
STAC file wildfire-pt-2025/cpm/S2C_MSIL2A_20250831T112131_N0511_R037_T29TPF_20250831T154413.json exists. Skipping.
s3://eodata/Sentinel-2/MSI/L2A/2025/08/30/S2A_MSIL2A_20250830T110701_N0511_R137_T29TPF_20250830T152014.SAFE
wildfire-pt-2025/cpm/S2A_MSIL2A_20250830T110701_N0511_R137_T29TPF_20250830T152014.zarr
https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-fra/cpm-manual/S2A_MSIL2A_20250830T110701_N0511_R137_T29TPF_20250830T152014.zarr/
STAC file wildfire-pt-2025/cpm/S2A_MSIL2A_20250830T110701_N0511_R137_T29TPF_20250830T152014.json exists. Skipping.
s3://eodata/Sentinel-2/MSI/L2A/2025/08/28/S2C_MSIL2A_20250828T110641_N0511_R137_T29T